<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson02/03_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: Convolutional neural network (CNN)

1. Python basics & the training loop
2. Dense neural network on FashionMNIST
3. **Convolutional neural network (CNN) on FashionMNIST** ← *you are here*
4. The same with fastai
5. Fine-tuning a pretrained model

The dense network from notebook 2 reached about 85% accuracy. But it has a weakness: `Flatten` turns the image into one long row of pixels. The network does not know which pixels are **neighbors**, and a pattern learned in one part of the image does not help in another part.

**Convolutional networks** fix this, and they power almost all image recognition. In this notebook you will:
- see what a **convolution** does, using hand-made **kernels** (edge detectors, blur) on real images
- build a CNN and see that, compared to notebook 2, **only the model definition changes**
- look at the kernels the CNN **learned by itself**

**Before you start:** *Runtime → Change runtime type → T4 GPU*. A CNN does much more computation than a dense network, so the GPU really helps here.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

The same data as in notebook 2:

In [ ]:
train_set = datasets.FashionMNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
test_set = datasets.FashionMNIST(root="data", train=False, download=True, transform=transforms.ToTensor())
class_names = train_set.classes

batch_size = 64
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size)

print("training images:", len(train_set), "  test images:", len(test_set))

---
## 1. What is a convolution?

A **kernel** (or *filter*) is a small grid of numbers, typically 3×3, that describes a **pattern**. This one looks for **vertical edges**: dark on the left, bright on the right.

In [ ]:
kernel = torch.tensor([[-1.0, 0.0, 1.0],
                       [-1.0, 0.0, 1.0],
                       [-1.0, 0.0, 1.0]])
kernel

We slide the kernel over the image. At every position, we multiply the 9 pixels under the kernel by the 9 kernel numbers and **add everything up**. A large number means *"the pattern is here"*.

First run the helper cell below. It only draws pictures, so you don't need to read it.

In [ ]:
# @title Helper functions (just run this cell)

def show_window(image, kernel, row, col):
    patch = image[row:row + 3, col:col + 3]
    value = (patch * kernel).sum().item()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image, cmap="gray")
    axes[0].add_patch(plt.Rectangle((col - 0.5, row - 0.5), 3, 3, edgecolor="red", facecolor="none", linewidth=2))
    axes[0].set_title(f"window at row {row}, column {col}")
    for ax, matrix, title, vmin in [(axes[1], patch, "pixels under the window", 0), (axes[2], kernel, "kernel", -1)]:
        ax.imshow(matrix, cmap="gray", vmin=vmin, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", color="red", fontsize=14)
        ax.set_title(title)
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"sum of (pixels × kernel) = {value:.2f}", fontsize=14)
    plt.show()

def apply_kernel(image, kernel):
    return F.conv2d(image.unsqueeze(0).unsqueeze(0), kernel.unsqueeze(0).unsqueeze(0), padding=1)[0, 0]

In [ ]:
image = train_set[1][0][0]                    # a T-shirt, 28x28 pixels
show_window(image, kernel, row=15, col=6)     # on the left edge of the shirt

In [ ]:
show_window(image, kernel, row=15, col=12)    # in the middle of the shirt

On the edge we get a large number, in the middle about 0. **The kernel found the edge.**

**Try it:** Move the window by changing `row` and `col` (0 to 25).

Doing this at *every* position gives a new image, called a **feature map**. Different kernels find different patterns:

In [ ]:
kernels = {
    "vertical edges":   torch.tensor([[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]]),
    "horizontal edges": torch.tensor([[-1.0, -1.0, -1.0], [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]]),
    "blur":             torch.ones(3, 3) / 9,
    "sharpen":          torch.tensor([[0.0, -1.0, 0.0], [-1.0, 5.0, -1.0], [0.0, -1.0, 0.0]]),
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("original")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    ax.imshow(apply_kernel(image, k), cmap="gray")
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.show()

Before deep learning, people designed kernels like these by hand. **A CNN learns its own kernels**: the 9 numbers are just parameters, trained by gradient descent like `m` and `b` in notebook 1.

Why is this better than a dense layer?
1. **The same kernel slides over the whole image**, so a pattern is found wherever it is.
2. **A kernel is tiny**: 9 weights, while each neuron of our dense network had 784.

---
## 2. The building blocks of a CNN

### Convolutional layer: `nn.Conv2d`

A convolutional layer holds **many kernels** at once. Each kernel produces one feature map (also called a *channel*):
- `in_channels=1`: our images are grayscale (a color image would have 3)
- `out_channels=32`: the layer learns 32 different kernels, so it produces 32 feature maps
- `kernel_size=3`: each kernel is 3×3
- `padding=1`: add a frame of zeros around the image, so that the output stays 28×28

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)

images, labels = next(iter(train_loader))
print("input: ", images.shape)             # [64 images, 1 channel, 28, 28]
print("output:", conv(images).shape)       # [64 images, 32 feature maps, 28, 28]

### Look inside: the kernels are just parameters

In [ ]:
print("weight shape:", conv.weight.shape)     # 32 kernels, each 1 x 3 x 3
print("bias shape:  ", conv.bias.shape)       # one bias per kernel
print("number of parameters:", sum(p.numel() for p in conv.parameters()))
print("\nthe first kernel (random for now):\n", conv.weight[0, 0])

Only 320 parameters (32 × 9 weights + 32 biases), yet the layer processes the whole image.

A **second** convolutional layer takes these 32 feature maps as input. Each of its kernels looks at all 32 maps at once (shape `32 × 3 × 3`) and combines simple patterns into more complex ones: edges → corners → collars, sleeves, heels...

### Max pooling: `nn.MaxPool2d`

After a convolution, we usually **shrink** the feature maps with **max pooling**: take each 2×2 block and keep only the largest number. The image becomes half as wide and half as high, but the strongest signals survive:

In [ ]:
tiny = torch.tensor([[1.0, 2.0, 0.0, 0.0],
                     [3.0, 4.0, 0.0, 1.0],
                     [0.0, 0.0, 5.0, 6.0],
                     [0.0, 1.0, 7.0, 8.0]])

pool = nn.MaxPool2d(kernel_size=2)
print(pool(tiny.unsqueeze(0)))

### Dropout: `nn.Dropout`

During training, **dropout** randomly switches off some neurons (here 25%) in each step. The network cannot rely on a single neuron and has to learn more robust patterns, which reduces **overfitting**. At test time dropout is turned off. That's why we call `model.train()` and `model.eval()`:

In [ ]:
dropout = nn.Dropout(0.25)
x = torch.ones(10)

dropout.train()
print("training mode:  ", dropout(x))    # some values set to 0, the rest scaled up to keep the average
dropout.eval()
print("evaluation mode:", dropout(x))    # nothing changes

---
## 3. The CNN

Here is the dense network from notebook 2 again, for comparison:

```python
class DenseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x
```

And here is the CNN. The pattern is the same (`__init__` lists the layers, `forward` says how the data flows). We just put two **convolution → ReLU → pooling** blocks *before* the flatten:

```
image 1×28×28 → conv: 32×28×28 → pool: 32×14×14 → conv: 64×14×14 → pool: 64×7×7 → flatten: 3136 → dense: 128 → dense: 10
```

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # [batch, 1, 28, 28]  -> [batch, 32, 14, 14]
        x = self.pool(self.relu(self.conv2(x)))   # -> [batch, 64, 7, 7]
        x = self.flatten(x)                       # -> [batch, 3136]
        x = self.dropout(self.relu(self.fc1(x)))  # -> [batch, 128]
        x = self.fc2(x)                           # -> [batch, 10]   one score per class
        return x

model = CNN().to(device)
model

### Look inside: follow one batch through the network

In [ ]:
x = images.to(device)
print("input:              ", x.shape)
x = model.pool(model.relu(model.conv1(x)))
print("after conv1 + pool: ", x.shape)
x = model.pool(model.relu(model.conv2(x)))
print("after conv2 + pool: ", x.shape)
x = model.flatten(x)
print("after flatten:      ", x.shape)
x = model.relu(model.fc1(x))
print("after fc1:          ", x.shape)
x = model.fc2(x)
print("after fc2:          ", x.shape)

### Look inside: how many parameters?

In [ ]:
for name, param in model.named_parameters():
    print(f"{name:14s} shape = {str(list(param.shape)):18s} count = {param.numel():,}")

total = sum(p.numel() for p in model.parameters())
conv_total = sum(p.numel() for name, p in model.named_parameters() if name.startswith("conv"))
print(f"\ntotal number of parameters: {total:,}")
print(f"of that in the convolutional layers: {conv_total:,}")

Surprised? Our CNN has *more* parameters than the dense network (109,386), but almost all of them sit in the dense layer `fc1`. The two convolutional layers, which do the actual "seeing", have fewer than 19,000 parameters!

---
## 4. Training

Now the best part. The helper functions and the training loop below are **copied from notebook 2 without any change**. The only thing that differs is the model.

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
    return correct / len(loader.dataset)

def show_prediction(model, image, label):
    model.eval()
    with torch.no_grad():
        scores = model(image.unsqueeze(0).to(device))
        probabilities = torch.softmax(scores, dim=1)[0].cpu()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    ax1.imshow(image[0], cmap="gray")
    ax1.set_title(f"true label: {class_names[label]}")
    ax1.axis("off")
    ax2.barh(class_names, probabilities)
    ax2.invert_yaxis()
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("predicted probability")
    plt.show()

print(f"test accuracy before training: {accuracy(model, test_loader):.1%}")

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
n_epochs = 5

train_losses, train_accuracies, test_accuracies = [], [], []

for epoch in range(n_epochs):
    start = time.time()
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        predictions = model(images)                   # 1. FORWARD
        loss = loss_fn(predictions, labels)           # 2. LOSS
        optimizer.zero_grad()                         # 5. RESET
        loss.backward()                               # 3. BACKWARD
        optimizer.step()                              # 4. UPDATE

        total_loss += loss.item()

    train_losses.append(total_loss / len(train_loader))
    train_accuracies.append(accuracy(model, train_loader))
    test_accuracies.append(accuracy(model, test_loader))
    print(f"epoch {epoch + 1}/{n_epochs}:  loss = {train_losses[-1]:.3f},  "
          f"train accuracy = {train_accuracies[-1]:.1%},  test accuracy = {test_accuracies[-1]:.1%}  "
          f"({time.time() - start:.0f} s)")

In [ ]:
epochs = range(1, n_epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_losses, "o-")
ax1.set_xlabel("epoch")
ax1.set_title("training loss")

ax2.plot(epochs, train_accuracies, "o-", label="train")
ax2.plot(epochs, test_accuracies, "o-", label="test")
ax2.set_xlabel("epoch")
ax2.set_title("accuracy")
ax2.legend()
plt.show()

Almost **90%**, compared to about 85% for the dense network, with the same data, the same loss, the same optimizer and the same number of epochs. Understanding neighboring pixels pays off!

In [ ]:
image, label = test_set[0]
show_prediction(model, image, label)

**Try it:** Look at other test images, e.g. `test_set[42]`. Can you find one the model gets wrong?

---
## 5. What did the CNN learn?

### Look inside: the learned kernels

These are the 32 kernels of the first layer. **Nobody designed them.** They started random and were shaped by gradient descent. Compare them with our hand-made kernels from section 1: several of them turned into edge detectors, blue (negative) on one side and red (positive) on the other, often diagonal!

In [ ]:
kernels_learned = model.conv1.weight.detach().cpu()     # shape [32, 1, 3, 3]
limit = kernels_learned.abs().max()

fig, axes = plt.subplots(4, 8, figsize=(10, 5.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(kernels_learned[i, 0], cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_title(f"kernel {i}", fontsize=8)
    ax.axis("off")
plt.show()

### Look inside: feature maps

What does the network "see"? Let's push one image through the convolutional layers and show the feature maps. Bright = the kernel found its pattern there.

In [ ]:
image, label = test_set[0]

model.eval()
with torch.no_grad():
    x = image.unsqueeze(0).to(device)
    maps1 = model.relu(model.conv1(x))                  # [1, 32, 28, 28]
    maps2 = model.relu(model.conv2(model.pool(maps1)))  # [1, 64, 14, 14]

def show_feature_maps(maps, title):
    fig, axes = plt.subplots(2, 8, figsize=(13, 3.5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(maps[0, i].cpu(), cmap="gray")
        ax.set_title(f"map {i}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.show()

plt.imshow(image[0], cmap="gray")
plt.title(f"input: {class_names[label]}")
plt.axis("off")
plt.show()

show_feature_maps(maps1, "layer 1: first 16 of 32 feature maps (28x28) - simple patterns like edges")
show_feature_maps(maps2, "layer 2: first 16 of 64 feature maps (14x14) - combinations of patterns")

The first layer produces maps that still look like the shoe, each highlighting different edges. The second layer's maps are coarser and more abstract. In deeper networks (like the one we will fine-tune in notebook 5), later layers respond to whole parts of objects: wheels, eyes, faces...

### Where does it still fail?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

all_predictions, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        all_predictions.append(model(images.to(device)).argmax(dim=1).cpu())
        all_labels.append(labels)
all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(all_labels, all_predictions, display_labels=class_names,
                                        xticks_rotation=45, colorbar=False, ax=ax)
plt.show()

*Shirt* is still the hardest class, but compare the diagonal with the confusion matrix in notebook 2.

---
## 6. Summary

| | dense network (notebook 2) | CNN (this notebook) |
|---|---|---|
| **first layers** | `Flatten` + `Linear`: every neuron sees all 784 pixels | `Conv2d`: small 3×3 kernels slide over the image |
| **knows about neighboring pixels** | no | yes |
| **a pattern found in one place helps elsewhere** | no | yes |
| **parameters** | 109,386 | 421,642 (only 18,816 in the conv layers) |
| **test accuracy after 5 epochs** | ~85% | ~89% |
| **loss, optimizer, training loop** | cross-entropy, SGD, 5 steps | **exactly the same** |

**New words:** kernel/filter, convolution, feature map, channel, padding, max pooling, dropout.

### Exercises
1. In section 1, design a kernel that detects **diagonal** edges and look at the result with `plt.imshow(apply_kernel(image, my_kernel), cmap="gray")`.
2. Change the number of kernels: `out_channels` of 16 and 32 instead of 32 and 64 (don't forget to change `in_channels` of `conv2` and the input size of `fc1`!). How do the parameter count and accuracy change?
3. Add a third block `conv3` (64 → 128 channels) + pooling. What is the image size after it, and what must the input size of `fc1` be now? *Hint: the shape-tracing cell from section 3 helps, and so does the error message if you get it wrong.* Did the number of parameters go up or down?
4. Train for 15 epochs, with and without dropout. Watch the gap between train and test accuracy.
5. **Bonus:** Try `torch.optim.Adam(model.parameters(), lr=0.001)`. Can you beat 91%?

**Next:** in notebook 4 we will do the same with **fastai**, a library that packs all of this (and many tricks) into a few lines.